# Week 10 (Live) — Multi-Agent System with LangGraph (Student)

## 0. Setup

In [ ]:
# HINT: %pip install -q langgraph langchain-openai langchain-tavily python-dotenv

# HINT: import os
# HINT: from dotenv import load_dotenv, find_dotenv
# HINT: call load_dotenv(find_dotenv(), override=True)
# HINT: read os.getenv("OPENAI_API_KEY") into api_key
# HINT: print whether the key loaded (yes/no) - don't print the actual key!

# HINT: from langchain_openai import ChatOpenAI
# HINT: create one shared llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


## 1. Shared State

In [ ]:
# HINT: from typing import TypedDict
# HINT: class AgentState(TypedDict): define exactly 3 fields ->
#       query: str          (the original user question)
#       research_data: str  (facts found by the Research Agent, "" if skipped)
#       final_output: str   (the finished answer from the Writer Agent)


## 2. The Search Tool (Research Agent's tool)

In [ ]:
# HINT: from langchain_core.tools import tool

# HINT: create MOCK_KNOWLEDGE_BASE - a dict of {topic: [list of fact strings]}
#       for 1-2 topics you expect to ask about

# HINT: write _mock_search(query) ->
#       - lowercase the query
#       - check which topics have any word overlapping with the query
#       - collect and de-duplicate the matching facts
#       - return them as a "\n"-joined bullet list (fall back to a friendly
#         "no mock data" message if nothing matched)

# HINT: check os.getenv("TAVILY_API_KEY") -> if present, try:
#       from langchain_tavily import TavilySearch
#       and create a client with max_results=5 (wrap in try/except ImportError)

# HINT: write an @tool-decorated function web_search_tool(query) that:
#       - if a real Tavily client was created, call it and pull out each
#         result's "content" field
#       - otherwise, fall back to _mock_search(query)

# HINT: test it - print(web_search_tool.invoke("some test query"))


## 3. Supervisor Node

In [ ]:
# HINT: define CONVERSATIONAL_PATTERNS - a list of greeting/small-talk words
#       like "hi", "hello", "thanks", "how are you"

# HINT: def supervisor_node(state):
#       - print a "Thought" message showing the query it received
#       - return {} (it does NOT change any state - it only routes)

# HINT: def route_after_supervisor(state) -> str:
#       - lowercase state["query"]
#       - check if it matches any CONVERSATIONAL_PATTERNS
#       - print your routing decision
#       - return "writer_agent" if conversational, else "research_agent"


## 4. Research Agent Node

In [ ]:
# HINT: def research_agent_node(state):
#       - print a "Thought" message using state["query"]
#       - call web_search_tool.invoke(state["query"])
#       - print which tool was called and what it returned
#       - return {"research_data": <the tool's result>}


## 5. Writer Agent Node

In [ ]:
# HINT: def writer_agent_node(state):
#       - has_research = bool(state["research_data"].strip())
#       - print a "Thought" message
#       - if has_research: build a prompt that says "use ONLY the research
#         notes below" and includes state["research_data"] + state["query"]
#       - else: build a simpler prompt using just state["query"]
#       - call llm.invoke(prompt)
#       - print that the LLM produced a final_output (and its length)
#       - return {"final_output": response.content}


## 6. Wire It Together: Build and Compile the Graph

In [ ]:
# HINT: from langgraph.graph import StateGraph, START, END
# HINT: builder = StateGraph(AgentState)

# HINT: builder.add_node(...) for "supervisor", "research_agent", "writer_agent"

# HINT: builder.add_edge(START, "supervisor")

# HINT: builder.add_conditional_edges(
#           "supervisor",
#           route_after_supervisor,
#           {"research_agent": "research_agent", "writer_agent": "writer_agent"},
#       )

# HINT: builder.add_edge("research_agent", "writer_agent")
# HINT: builder.add_edge("writer_agent", END)

# HINT: graph = builder.compile()

# HINT: try/except: print(graph.get_graph().draw_mermaid())


## 7. Run It

In [ ]:
# HINT: build initial_state matching AgentState:
#       - query = your own example question
#       - research_data = ""
#       - final_output = ""

# HINT: result = graph.invoke(initial_state)

# HINT: print result["final_output"]

# TRY THIS: change the query to something conversational (like "hi") and
# re-run - which path does the Supervisor take this time?


## 8. Turn It Into a Chatbot (while loop)

In [ ]:
# HINT: EXIT_WORDS = {"exit", "quit", "bye"}
# HINT: print a welcome message

# HINT: while True:
#       - wrap input("You: ") in try/except so the cell exits cleanly if no
#         keyboard is attached (e.g. an automated "run all cells")
#       - skip empty input with continue
#       - break out of the loop if the user typed an exit word
#       - build a FRESH chat_state (matching AgentState) every single loop -
#         remember, this version has no memory between turns
#       - result = graph.invoke(chat_state)
#       - print the bot's result["final_output"]


## Next Steps (for when we "improvise")

- Swap `web_search_tool`'s mock fallback for a real search API.
- Make the Supervisor's routing decision with the LLM instead of a keyword list.
- Add a `Critique` node that checks the Writer Agent's answer.
- Give the chatbot real memory across turns (without changing `AgentState`'s 3 fields).